In [1]:
from tensorflow.keras import Model
from tensorflow.keras.layers import GlobalMaxPooling1D, Concatenate

In [3]:
# -----------------------------
# 1. Imports
# -----------------------------
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, f1_score, cohen_kappa_score,
    recall_score, precision_score, roc_auc_score,
    matthews_corrcoef
)

from tensorflow.keras import Model
from tensorflow.keras.layers import (
    Input, Conv1D, MaxPooling1D, GlobalMaxPooling1D,
    LSTM, Dense, Dropout, Concatenate
)
from tensorflow.keras.callbacks import EarlyStopping

In [4]:
# -----------------------------
# 2. Load Wisconsin Breast Cancer Diagnostic dataset
# -----------------------------
data = load_breast_cancer()

X = data.data
y = data.target

# In sklearn's breast cancer dataset:
# 0 = malignant
# 1 = benign
print(data.target_names)
print("X shape:", X.shape)
print("y shape:", y.shape)

['malignant' 'benign']
X shape: (569, 30)
y shape: (569,)


In [5]:
# -----------------------------
# 3. Train / validation / test split
# -----------------------------
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.25,
    stratify=y_trainval,
    random_state=42
)

# 60% train, 20% validation, 20% test
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (341, 30)
Validation: (114, 30)
Test: (114, 30)


In [6]:
# -----------------------------
# 4. Scale and reshape data
# -----------------------------
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

n_features = X_train_scaled.shape[1]

# Reshape for CNN/LSTM input: samples, timesteps, channels
X_train_seq = X_train_scaled.reshape((X_train_scaled.shape[0], n_features, 1))
X_val_seq = X_val_scaled.reshape((X_val_scaled.shape[0], n_features, 1))
X_test_seq = X_test_scaled.reshape((X_test_scaled.shape[0], n_features, 1))

print("CNN-LSTM input shape:", X_train_seq.shape)

CNN-LSTM input shape: (341, 30, 1)


In [7]:
# -----------------------------
# 5. Build CNN-LSTM model
# -----------------------------
tf.keras.utils.set_random_seed(42)

inputs = Input(shape=(n_features, 1))

# CNN branch
cnn_branch = Conv1D(filters=32, kernel_size=3, padding="same", activation="relu")(inputs)
cnn_branch = MaxPooling1D(pool_size=2)(cnn_branch)
cnn_branch = Conv1D(filters=64, kernel_size=3, padding="same", activation="relu")(cnn_branch)
cnn_branch = GlobalMaxPooling1D()(cnn_branch)

# LSTM branch
lstm_branch = LSTM(32)(inputs)

# Combine CNN and LSTM outputs
combined = Concatenate()([cnn_branch, lstm_branch])

x = Dense(32, activation="relu")(combined)
x = Dropout(0.3)(x)
outputs = Dense(1, activation="sigmoid")(x)

cnn_lstm_model = Model(inputs=inputs, outputs=outputs)

cnn_lstm_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy", tf.keras.metrics.AUC(name="auc")]
)

cnn_lstm_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 30, 1)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 30, 32)    │        128 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 15, 32)    │          0 │ conv1d[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 15, 64)    │      6,208 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 64)        │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 32)        │      4,352 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 96)        │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 32)        │      3,104 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 32)        │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 1)         │         33 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 13,825 (54.00 KB)

 Trainable params: 13,825 (54.00 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
# -----------------------------
# 6. Train CNN-LSTM model
# -----------------------------
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history = cnn_lstm_model.fit(
    X_train_seq, y_train,
    validation_data=(X_val_seq, y_val),
    epochs=60,
    batch_size=32,
    verbose=1,
    callbacks=[early_stop]
)

Epoch 1/60
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.7889 - auc: 0.8728 - loss: 0.5978 - val_accuracy: 0.8509 - val_auc: 0.9523 - val_loss: 0.5072
Epoch 2/60
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9032 - auc: 0.9541 - loss: 0.4040 - val_accuracy: 0.8421 - val_auc: 0.9464 - val_loss: 0.3822
Epoch 3/60
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9003 - auc: 0.9431 - loss: 0.3112 - val_accuracy: 0.8509 - val_auc: 0.9496 - val_loss: 0.3307
Epoch 4/60
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9032 - auc: 0.9446 - loss: 0.2824 - val_accuracy: 0.8509 - val_auc: 0.9525 - val_loss: 0.3180
Epoch 5/60
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9062 - auc: 0.9629 - loss: 0.2421 - val_accuracy: 0.8772 - val_auc: 0.9517 - val_loss: 0.3103
Epoch 6/60
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9179 - auc: 0.9546 - loss: 0.2510 - val_accuracy: 0.8772 - val_auc: 0.9571 - val_loss: 0.2921
Epoch 7/60
11/11 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step -

In [9]:
# -----------------------------
# 7. Evaluate model
# -----------------------------
train_loss, train_acc, train_auc = cnn_lstm_model.evaluate(X_train_seq, y_train, verbose=0)
val_loss, val_acc, val_auc = cnn_lstm_model.evaluate(X_val_seq, y_val, verbose=0)

y_test_prob = cnn_lstm_model.predict(X_test_seq, verbose=0).ravel()
y_test_pred = (y_test_prob >= 0.5).astype(int)

# Class labels
POS_LABEL = 0   # malignant
NEG_LABEL = 1   # benign

test_acc = accuracy_score(y_test, y_test_pred)
f1 = f1_score(y_test, y_test_pred, pos_label=POS_LABEL)
kappa = cohen_kappa_score(y_test, y_test_pred)
mcc = matthews_corrcoef(y_test, y_test_pred)
recall = recall_score(y_test, y_test_pred, pos_label=POS_LABEL)
specificity = recall_score(y_test, y_test_pred, pos_label=NEG_LABEL)
precision = precision_score(y_test, y_test_pred, pos_label=POS_LABEL)
auc = roc_auc_score(y_test, y_test_prob)

cnn_lstm_results = pd.DataFrame([{
    "Model": "CNN-LSTM",
    "Train_accuracy": round(train_acc, 4),
    "Val_accuracy": round(val_acc, 4),
    "Test_accuracy": round(test_acc, 4),
    "F1_score": round(f1, 4),
    "Kappa": round(kappa, 4),
    "MCC": round(mcc, 4),
    "Recall": round(recall, 4),
    "Specificity": round(specificity, 4),
    "Precision": round(precision, 4),
    "AUC": round(auc, 4)
}])

print(cnn_lstm_results.to_string(index=False))

   Model  Train_accuracy  Val_accuracy  Test_accuracy  F1_score  Kappa    MCC  Recall  Specificity  Precision    AUC
CNN-LSTM          0.9941        0.9649         0.9474    0.9302  0.888 0.8886  0.9524       0.9444     0.9091 0.9868
